# AgriProcure — Interactive Analytics Dashboard
**From Field to Forecast: weather-linked price analytics for tomato, onion and potato mandis**

One notebook, twelve interactive tabs, every number read live from the repo's own `data/` and `reports/` files.

| Tab | What you can do |
|---|---|
| 1 Overview | KPI cards, price index, a plain "what the evidence says" summary |
| 2 Market Explorer | Filter crop / state / mandi / dates, overlay rain, temperature or arrivals |
| 3 Map & Heatmap | Mandi map (real weather-matching coordinates), state × week heatmaps |
| 4 Weather & Supply Drivers | Scatter, binned effects and Spearman tests of next-week price change |
| 5 Forecast Lab | SARIMA vs baselines, rolling-origin backtest, wins per fold, 4-week fan chart |
| 6 Direction Models | Rise / fall / flat classifiers, baselines, per-class recall, saved figures |
| 7 Rules Explorer | Filter association rules, flag trivial / redundant ones, cross-crop check |
| 8 Spike-Risk Scorer | Tick the conditions you observe, see which rules fire |
| 9 Procurement Planner | Buy-now vs wait using forecast and its 80 % interval |
| 10 Data & Methods QA | Coverage, stationarity, residuals, VIF, data caveats |
| 11 Literature Gaps | Which gaps the project closes, partly closes, or leaves open |
| 12 Export | Save filtered tables and an HTML snapshot |

**How to run:** put this file in `AgriProcure-main/notebooks/`, then `pip install pandas numpy scipy plotly ipywidgets` and *Run All*. Works in JupyterLab, classic Notebook and VS Code (Colab: enable custom widget manager).

In [ ]:
# ── 1. Setup ──────────────────────────────────────────────────────────────────────────────
import warnings, re, json
from pathlib import Path
import numpy as np, pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import ipywidgets as W
from IPython.display import display, Image, clear_output
from scipy import stats

warnings.filterwarnings("ignore")
pd.set_option("display.max_colwidth", 140)
px.defaults.template = "plotly_white"

def HTML(s):
    """Raw-HTML widget (works both inside layouts and with display())."""
    return W.HTML(value=s)

def find_root():
    here = Path.cwd().resolve()
    for base in [here, *here.parents]:
        for cand in (base, base / "AgriProcure-main", base / "AgriProcure", base / "AgriProcure" / "AgriProcure"):
            if (cand / "data" / "panel_with_weather.csv").exists():
                return cand
    raise FileNotFoundError("Open this notebook from inside the AgriProcure repo (needs data/panel_with_weather.csv).")

ROOT = find_root(); DATA = ROOT / "data"; REP = ROOT / "reports"; EXPORT = ROOT / "exports"
EXPORT.mkdir(parents=True, exist_ok=True)  # create early so any tab can write CSVs
print("Repo root:", ROOT)

COL = {"Tomato": "#d9480f", "Onion": "#7048e8", "Potato": "#a67c00"}
CROPS = list(COL)
MODEL_COL = {"Naive": "#868e96", "Seasonal_Naive": "#339af0", "SARIMA_Univariate": "#2f9e44", "SARIMAX_Weather": "#e03131"}
MODEL_LBL = {"Naive": "Naive (last week)", "Seasonal_Naive": "Seasonal naive (13 wk)",
             "SARIMA_Univariate": "SARIMA", "SARIMAX_Weather": "SARIMAX + weather"}

display(HTML("""<style>
.ap-kpis{display:flex;flex-wrap:wrap;gap:10px;margin:6px 0 10px}
.ap-kpi{flex:1 1 150px;min-width:150px;padding:10px 14px;border-radius:10px;border:1px solid rgba(128,128,128,.28);background:rgba(128,128,128,.07)}
.ap-kl{font-size:11.5px;text-transform:uppercase;letter-spacing:.04em;opacity:.7}
.ap-kv{font-size:24px;font-weight:700;line-height:1.25}
.ap-ks{font-size:11.5px;opacity:.7}
.ap-call{padding:9px 13px;border-radius:8px;margin:6px 0;border-left:5px solid;font-size:13.5px;line-height:1.45}
.ap-info{border-color:#339af0;background:rgba(51,154,240,.10)}
.ap-warn{border-color:#f08c00;background:rgba(240,140,0,.12)}
.ap-bad{border-color:#e03131;background:rgba(224,49,49,.10)}
.ap-good{border-color:#2f9e44;background:rgba(47,158,68,.10)}
table.ap-t{border-collapse:collapse;width:100%;font-size:13px}
table.ap-t th{text-align:left;padding:7px 9px;border-bottom:2px solid rgba(128,128,128,.4)}
table.ap-t td{padding:7px 9px;border-bottom:1px solid rgba(128,128,128,.22);vertical-align:top}
.ap-badge{display:inline-block;padding:1px 9px;border-radius:10px;font-size:11.5px;font-weight:600;color:#fff;white-space:nowrap}
.ap-h1{font-size:22px;font-weight:700;margin:2px 0}
.ap-sub{opacity:.7;font-size:13px;margin-bottom:6px}
</style>"""))

# ── small UI helpers ──
def kpis(items):
    cells = "".join(f"<div class='ap-kpi'><div class='ap-kl'>{l}</div><div class='ap-kv' style='color:{c}'>{v}</div><div class='ap-ks'>{s}</div></div>"
                    for l, v, s, c in [(*i, "inherit")[:4] if len(i) < 4 else i for i in items])
    return HTML(f"<div class='ap-kpis'>{cells}</div>")

def callout(text, kind="info"):
    return HTML(f"<div class='ap-call ap-{kind}'>{text}</div>")

def pct(x, d=1): return f"{x:.{d}f}%"
def inr(x): return f"₹{x:,.0f}"

def out(): return W.Output(layout=W.Layout(width="100%"))
def show(o, *objs):
    with o:
        clear_output(wait=True)
        for x in objs: display(x)

def watch(widgets, fn):
    def _h(ch):
        if ch["name"] == "value": fn()
    for w in widgets: w.observe(_h, names="value")

def table(df, fmt=None, gradient=None):
    """Pretty DataFrame (Styler if available, plain otherwise)."""
    try:
        s = df.style.hide(axis="index")
        if fmt: s = s.format(fmt)
        if gradient: s = s.background_gradient(subset=gradient, cmap="RdYlGn_r")
        return s
    except Exception:
        return df

FIGS = {}      # last figure per tab, used by the Export tab
STATE = {}     # last filtered tables, used by the Export tab
LAYOUT_ROW = W.Layout(flex_flow="row wrap", align_items="center", width="100%")

In [ ]:
# ── 2. Load every dataset the dashboard uses ─────────────────────────────────────────────
panel = pd.read_csv(DATA / "panel_with_weather.csv", parse_dates=["week_start_date"])
feats = pd.read_csv(DATA / "features_panel.csv", parse_dates=["week_start_date"])
bt    = pd.read_csv(DATA / "sarima_forecasts_vs_actuals.csv", parse_dates=["Forecast_Date"])
fut   = pd.read_csv(DATA / "sarima_future_forecasts.csv", parse_dates=["Week_Start_Date"])
rules = pd.read_csv(DATA / "association_rules.csv")
rcv   = pd.read_csv(REP / "association_rules_crop_cross_validation.csv")
sar_eval = pd.read_csv(REP / "sarima_evaluation_metrics.csv")
stat_tbl = pd.read_csv(REP / "sarima_stationarity_tests.csv")
resid_tbl = pd.read_csv(REP / "sarima_residual_diagnostics.csv")
mres  = pd.read_csv(REP / "model_results.csv")
bin5  = pd.read_csv(REP / "binary_prediction_metrics_5pct.csv")
dlml  = pd.read_csv(REP / "dl_and_ml_classification_metrics_5pct.csv")
vif   = pd.read_csv(REP / "feature_vif.csv").rename(columns={"Unnamed: 0": "feature"})
coords = (pd.read_csv(DATA / "weather_daily_by_mandi.csv", usecols=["market_name", "latitude", "longitude"])
            .drop_duplicates("market_name"))
dim_mandi_path = DATA / "powerbi_exports" / "dim_mandi.csv"

# national weekly series (same construction as the SARIMA notebook: mean over mandis)
nat = (panel.groupby(["crop", "week_start_date"])
            .agg(price=("weekly_modal_price", "mean"), arrivals=("weekly_total_arrivals", "sum"),
                 rain=("weekly_rainfall_mm", "mean"), temp=("weekly_temp_mean_c", "mean")).reset_index())
WEEKS = sorted(panel.week_start_date.unique())
N_MANDI, N_STATE, N_WEEKS = panel.market_name.nunique(), panel.state.nunique(), len(WEEKS)

# ── forecast analytics ──
bt["abs_pct"] = bt.Percent_Error.abs()
piv = bt.pivot_table(index=["Crop", "Horizon", "Origin_Week"], columns="Model", values="Absolute_Error").reset_index()
piv["win_vs_snaive"] = piv["SARIMA_Univariate"] < piv["Seasonal_Naive"]
piv["win_vs_naive"]  = piv["SARIMA_Univariate"] < piv["Naive"]
mape_tbl = bt.groupby(["Crop", "Model", "Horizon"]).agg(MAE=("Absolute_Error", "mean"), MAPE=("abs_pct", "mean"), Folds=("Origin_Week", "nunique")).reset_index()
mean_mape = bt.groupby("Model").abs_pct.mean()

# ── rules analytics: split antecedents, flag trivial and redundant rules ──
rules["items"] = rules.Antecedents.str.split(r"\s*\+\s*", regex=True).apply(frozenset)
rules["n_items"] = rules["items"].apply(len)
SPREAD_SHARE = float((feats.price_spread_pct > 15).mean())          # High_Price_Spread threshold = 15 %
TRIVIAL_ITEMS = {"High_Price_Spread"} if SPREAD_SHARE >= 0.95 else set()
rules["trivial"] = rules["items"].apply(lambda s: bool(s & TRIVIAL_ITEMS))
rules["baseline_p"] = rules.Confidence / rules.Lift                  # P(outcome) implied by lift

def _redundant(df):
    flag = pd.Series(False, index=df.index)
    for _, g in df.groupby(["Crop", "Target_Outcome"]):
        idx = list(g.index); its = g["items"].tolist(); cf = g["Confidence"].tolist()
        for a in range(len(idx)):
            for b in range(len(idx)):
                if a != b and its[b] < its[a] and cf[b] >= cf[a]:
                    flag[idx[a]] = True; break
    return flag
rules["redundant"] = _redundant(rules)
ALL_ITEMS = sorted(set().union(*rules["items"]))
WEATHER_ITEMS = {i for i in ALL_ITEMS if re.search(r"Rain|Heat|Humid", i)}
rules["weather_rule"] = rules["items"].apply(lambda s: bool(s & WEATHER_ITEMS))

# cross-crop validation: does each rule still beat its own base rate in each crop?
rcv["baseline_p"] = rcv.Global_Confidence / rcv.Global_Lift
rcv["holds"] = rcv.Empirical_Confidence > rcv.baseline_p
rcv["gap_pts"] = (rcv.Empirical_Confidence - rcv.Global_Confidence) * 100

print(f"panel {panel.shape} | features {feats.shape} | backtest {bt.shape} | rules {rules.shape} "
      f"| trivial rules {int(rules.trivial.sum())} | redundant {int(rules.redundant.sum())}")

## Tab builders

In [ ]:
# ── TAB 1 · Overview ─────────────────────────────────────────────────────────────────────
def build_overview():
    uni = mean_mape.get("SARIMA_Univariate", np.nan); nv = mean_mape.get("Naive", np.nan)
    sn = mean_mape.get("Seasonal_Naive", np.nan); sx = mean_mape.get("SARIMAX_Weather", np.nan)
    wins_sn = piv.win_vs_snaive.mean() * 100
    ml = mres[~mres.experiment.str.contains("Baselines")]
    best = ml.sort_values("mean_macro_f1", ascending=False).iloc[0]
    base = mres[mres.experiment.str.contains("Baselines")].mean_macro_f1.max()
    n_ok = int((~rules.trivial & ~rules.redundant).sum())

    cards = kpis([
        ("Mandis", f"{N_MANDI}", f"{N_STATE} states", "#1c7ed6"),
        ("History", f"{N_WEEKS} wks", f"{pd.Timestamp(WEEKS[0]):%d %b %Y} → {pd.Timestamp(WEEKS[-1]):%d %b %Y}", "#1c7ed6"),
        ("Panel rows", f"{len(panel):,}", "market × crop × week", "#1c7ed6"),
        ("SARIMA MAPE", pct(uni, 2), f"naive {pct(nv, 2)} · seasonal-naive {pct(sn, 2)}", "#2f9e44"),
        ("Best direction model", f"{best.mean_macro_f1:.2f}", f"macro-F1 · {best.baseline} (baseline {base:.2f})", "#2f9e44"),
        ("Association rules", f"{len(rules)}", f"{n_ok} after dropping trivial / redundant", "#7048e8"),
        ("Max lift", f"{rules.Lift.max():.2f}×", f"median {rules.Lift.median():.2f}×", "#7048e8"),
    ])

    idx = nat.copy(); idx["index"] = idx.groupby("crop").price.transform(lambda s: s / s.iloc[0] * 100)
    fig = make_subplots(rows=1, cols=2, column_widths=[.62, .38], subplot_titles=("National weekly price index (first week = 100)", "Mean abs. % error by model, 1–4 week horizons"))
    for c in CROPS:
        d = idx[idx.crop == c]; fig.add_trace(go.Scatter(x=d.week_start_date, y=d["index"], name=c, line=dict(color=COL[c], width=2.4)), 1, 1)
    order = ["Naive", "Seasonal_Naive", "SARIMA_Univariate", "SARIMAX_Weather"]
    fig.add_trace(go.Bar(x=[MODEL_LBL[m] for m in order], y=[mean_mape[m] for m in order], marker_color=[MODEL_COL[m] for m in order],
                         text=[f"{mean_mape[m]:.2f}%" for m in order], textposition="outside", showlegend=False), 1, 2)
    fig.update_yaxes(title_text="Index", row=1, col=1); fig.update_yaxes(title_text="MAPE %", row=1, col=2, type="log")
    fig.update_layout(height=380, margin=dict(t=50, b=20), legend=dict(orientation="h", y=-0.15)); FIGS["overview"] = fig

    notes = [
        callout(f"<b>Forecasting.</b> SARIMA beats the naive random walk clearly ({pct(uni,2)} vs {pct(nv,2)} MAPE). "
                f"Against the <i>seasonal-naive</i> baseline it is nearly tied ({pct(uni,2)} vs {pct(sn,2)}); it had the smaller error in "
                f"<b>{wins_sn:.0f}%</b> of {len(piv)} crop × horizon × fold comparisons. With only 8 folds that is not strong evidence of an edge.", "info"),
        callout(f"<b>Weather as a linear regressor failed.</b> SARIMAX with rain / temperature / arrivals averaged <b>{pct(sx,1)}</b> MAPE vs {pct(uni,2)} for plain SARIMA. "
                "Weather is better treated as event triggers (association rules) than as a linear regressor on this sample.", "bad"),
        callout(f"<b>Direction model.</b> Best tuned model reaches macro-F1 {best.mean_macro_f1:.2f} ± {best.std_macro_f1:.2f} across time-series CV folds "
                f"(persistence baseline {base:.2f}). The spread across folds is as large as the gaps between the top models, so treat the ranking as indicative.", "warn"),
        callout(f"<b>Rules.</b> Lifts range {rules.Lift.min():.2f}–{rules.Lift.max():.2f}×: real but modest signal. "
                f"{int(rules.trivial.sum())} rules include <code>High_Price_Spread</code>, which is true in {SPREAD_SHARE*100:.0f}% of weeks (adds no information); {int(rules.redundant.sum())} are redundant.", "warn"),
        callout(f"<b>Scale.</b> {N_WEEKS} weeks is a single annual cycle, so the 13-week seasonality cannot be confirmed across years and no out-of-time test of the rules exists yet.", "warn"),
    ]
    o = out(); show(o, cards, fig, *notes)
    return o

# ── TAB 2 · Market Explorer ──────────────────────────────────────────────────────────────
def build_market():
    crop = W.ToggleButtons(options=CROPS, description="Crop", style={"button_width": "80px"})
    state = W.Dropdown(options=["All states"] + sorted(panel.state.unique()), description="State", layout=W.Layout(width="260px"))
    mandi = W.Dropdown(options=["All mandis (average)"], description="Mandi", layout=W.Layout(width="360px"))
    wk = W.SelectionRangeSlider(options=[(pd.Timestamp(w).strftime("%d %b %y"), pd.Timestamp(w)) for w in WEEKS],
                                index=(0, len(WEEKS) - 1), description="Weeks", layout=W.Layout(width="640px"))
    overlay = W.Dropdown(options=["Rainfall (mm)", "Total arrivals (MT)", "Temperature (°C)", "Humidity (%)", "None"], description="Overlay", layout=W.Layout(width="260px"))
    ma = W.IntSlider(value=1, min=1, max=8, description="Smooth (wk)", layout=W.Layout(width="260px"))
    band = W.Checkbox(value=True, description="Min–max band", indent=False)
    flags = W.Checkbox(value=True, description="Mark heavy-rain weeks (>35 mm)", indent=False)
    excl = W.Checkbox(value=False, description="Exclude partial weeks", indent=False)
    rank_by = W.Dropdown(options=["Mean price", "Price volatility (CV %)", "Mean weekly arrivals", "Mean weekly rainfall"], description="Rank mandis by", layout=W.Layout(width="320px"))
    o_main, o_rank = out(), out()

    def base():
        d = panel[panel.crop == crop.value]
        if state.value != "All states": d = d[d.state == state.value]
        lo, hi = wk.value; d = d[(d.week_start_date >= lo) & (d.week_start_date <= hi)]
        if excl.value: d = d[~d.is_partial_week]
        return d

    def refresh_mandi(*_):
        d = panel[(panel.crop == crop.value)]
        if state.value != "All states": d = d[d.state == state.value]
        opts = ["All mandis (average)"] + sorted(d.market_name.unique())
        keep = mandi.value if mandi.value in opts else opts[0]
        def _mandi_render(change):
            if change["name"] == "value": render()
        mandi.unobserve_all("value"); mandi.options = opts; mandi.value = keep; mandi.observe(_mandi_render, names="value")

    def render():
        d = base()
        if not mandi.value.startswith("All"): d = d[d.market_name == mandi.value]
        if d.empty: show(o_main, callout("No data for this selection.", "warn")); return
        g = d.groupby("week_start_date").agg(price=("weekly_modal_price", "mean"), lo=("weekly_min_price", "mean"), hi=("weekly_max_price", "mean"),
                                             arr=("weekly_total_arrivals", "sum"), rain=("weekly_rainfall_mm", "mean"),
                                             temp=("weekly_temp_mean_c", "mean"), hum=("weekly_humidity_mean_pct", "mean")).reset_index()
        g["ma"] = g.price.rolling(ma.value, min_periods=1).mean()
        fig = make_subplots(specs=[[{"secondary_y": True}]]); c = COL[crop.value]
        if band.value:
            fig.add_trace(go.Scatter(x=g.week_start_date, y=g.hi, line=dict(width=0), hoverinfo="skip", showlegend=False))
            fig.add_trace(go.Scatter(x=g.week_start_date, y=g.lo, fill="tonexty", fillcolor="rgba(128,128,128,.15)", line=dict(width=0), name="Min–max range", hoverinfo="skip"))
        ov = {"Rainfall (mm)": ("rain", "bar"), "Total arrivals (MT)": ("arr", "bar"), "Temperature (°C)": ("temp", "line"), "Humidity (%)": ("hum", "line")}.get(overlay.value)
        if ov:
            col, kind = ov
            tr = go.Bar(x=g.week_start_date, y=g[col], name=overlay.value, marker_color="rgba(51,154,240,.35)") if kind == "bar" else \
                 go.Scatter(x=g.week_start_date, y=g[col], name=overlay.value, line=dict(color="#339af0", dash="dot"))
            fig.add_trace(tr, secondary_y=True)
        fig.add_trace(go.Scatter(x=g.week_start_date, y=g.price, name="Modal price", line=dict(color=c, width=2.6)), secondary_y=False)
        if ma.value > 1: fig.add_trace(go.Scatter(x=g.week_start_date, y=g.ma, name=f"{ma.value}-wk average", line=dict(color="black", dash="dash", width=1.5)), secondary_y=False)
        if flags.value:
            h = g[g.rain > 35]
            if len(h): fig.add_trace(go.Scatter(x=h.week_start_date, y=h.price, mode="markers", name="Heavy-rain week", marker=dict(symbol="x", size=9, color="#1864ab")), secondary_y=False)
        who = mandi.value if not mandi.value.startswith("All") else (state.value if state.value != "All states" else "all mandis")
        fig.update_yaxes(title_text="₹ / quintal", secondary_y=False); fig.update_yaxes(title_text=overlay.value if ov else "", showgrid=False, secondary_y=True)
        fig.update_layout(height=470, title=f"{crop.value} — {who}", hovermode="x unified", margin=dict(t=55, b=20), legend=dict(orientation="h", y=-0.15))
        FIGS["market"] = fig; STATE["market_weekly"] = g
        sd = g.price.std(); cv = sd / g.price.mean() * 100
        ch4 = (g.price.iloc[-1] / g.price.iloc[-5] - 1) * 100 if len(g) >= 5 else np.nan
        r, p = stats.pearsonr(g.price, g.rain) if g.rain.std() > 0 and len(g) > 4 else (np.nan, np.nan)
        k = kpis([("Mean price", inr(g.price.mean()), "₹/quintal", COL[crop.value]), ("Volatility (CV)", pct(cv), f"σ = {inr(sd)}", "inherit"),
                  ("Latest", inr(g.price.iloc[-1]), f"{g.week_start_date.iloc[-1]:%d %b %Y}", "inherit"),
                  ("4-week change", "–" if np.isnan(ch4) else f"{ch4:+.1f}%", "latest vs 4 wks earlier", "inherit"),
                  ("Peak", inr(g.price.max()), f"{g.loc[g.price.idxmax(), 'week_start_date']:%d %b}", "inherit"),
                  ("Price ↔ rain (Pearson)", "–" if np.isnan(r) else f"{r:+.2f}", "" if np.isnan(p) else f"p = {p:.3f}, n = {len(g)}", "inherit")])
        show(o_main, k, fig); render_rank()

    def render_rank():
        d = base()
        g = d.groupby(["market_name", "state"]).agg(mean_price=("weekly_modal_price", "mean"), sd=("weekly_modal_price", "std"),
                                                    arr=("weekly_total_arrivals", "mean"), rain=("weekly_rainfall_mm", "mean")).reset_index()
        g["cv"] = g.sd / g.mean_price * 100
        key = {"Mean price": "mean_price", "Price volatility (CV %)": "cv", "Mean weekly arrivals": "arr", "Mean weekly rainfall": "rain"}[rank_by.value]
        g = g.sort_values(key, ascending=False).head(15).sort_values(key)
        fig = px.bar(g, x=key, y="market_name", color="state", orientation="h", height=max(320, 26 * len(g) + 90),
                     labels={key: rank_by.value, "market_name": ""}, title=f"Top mandis by {rank_by.value.lower()} (current filters)")
        fig.update_layout(margin=dict(t=50, b=20), legend=dict(orientation="h", y=-0.2)); STATE["market_rank"] = g
        show(o_rank, fig)

    watch([crop, state], refresh_mandi)
    watch([crop, state, wk, overlay, ma, band, flags, excl, mandi], render); watch([rank_by], render_rank)
    refresh_mandi(); render()
    row1 = W.HBox([crop, state, mandi], layout=LAYOUT_ROW); row2 = W.HBox([wk, overlay, ma], layout=LAYOUT_ROW)
    row3 = W.HBox([band, flags, excl], layout=LAYOUT_ROW)
    return W.VBox([row1, row2, row3, o_main, W.HBox([rank_by]), o_rank])

# ── TAB 3 · Map & Heatmap ────────────────────────────────────────────────────────────────
def build_map():
    crop = W.ToggleButtons(options=CROPS, description="Crop", style={"button_width": "80px"})
    metric = W.Dropdown(options=["Mean price (₹/qtl)", "Volatility (CV %)", "Price change over period (%)", "Mean weekly arrivals (MT)", "Mean weekly rainfall (mm)"], description="Map colour", layout=W.Layout(width="340px"))
    wk = W.SelectionRangeSlider(options=[(pd.Timestamp(w).strftime("%d %b %y"), pd.Timestamp(w)) for w in WEEKS], index=(0, len(WEEKS) - 1), description="Weeks", layout=W.Layout(width="640px"))
    hv = W.Dropdown(options=["Price (₹/qtl)", "Price vs state average (%)", "Rainfall (mm)", "Total arrivals (MT)"], description="Heatmap", layout=W.Layout(width="340px"))
    o_map, o_heat = out(), out()

    def render():
        lo, hi = wk.value
        d = panel[(panel.crop == crop.value) & (panel.week_start_date >= lo) & (panel.week_start_date <= hi)]
        g = d.groupby(["market_name", "state"]).apply(lambda x: pd.Series({
            "Mean price (₹/qtl)": x.weekly_modal_price.mean(), "Volatility (CV %)": x.weekly_modal_price.std() / x.weekly_modal_price.mean() * 100,
            "Price change over period (%)": (x.sort_values("week_start_date").weekly_modal_price.iloc[-1] / x.sort_values("week_start_date").weekly_modal_price.iloc[0] - 1) * 100,
            "Mean weekly arrivals (MT)": x.weekly_total_arrivals.mean(), "Mean weekly rainfall (mm)": x.weekly_rainfall_mm.mean()}),
          include_groups=False).reset_index().merge(coords, on="market_name", how="left")
        m = metric.value
        scale = "RdBu_r" if "change" in m else "Turbo"
        fig = px.scatter_geo(g, lat="latitude", lon="longitude", color=m, size="Mean weekly arrivals (MT)", hover_name="market_name",
                             hover_data={"state": True, "latitude": False, "longitude": False, "Mean price (₹/qtl)": ":.0f", "Volatility (CV %)": ":.1f", "Mean weekly arrivals (MT)": ":.0f"},
                             color_continuous_scale=scale, color_continuous_midpoint=0 if "change" in m else None, size_max=26, height=560,
                             title=f"{crop.value}: {m} (bubble size = mean weekly arrivals)")
        fig.update_geos(scope="asia", fitbounds="locations", showcountries=True, countrycolor="#adb5bd", showland=True, landcolor="rgba(200,200,200,.18)", showocean=True, oceancolor="rgba(120,170,220,.12)")
        fig.update_layout(margin=dict(t=50, b=0)); FIGS["map"] = fig; STATE["map_table"] = g
        show(o_map, fig)
        # heatmap
        dd = panel[panel.crop == crop.value]
        col = {"Price (₹/qtl)": "weekly_modal_price", "Price vs state average (%)": "weekly_modal_price", "Rainfall (mm)": "weekly_rainfall_mm", "Total arrivals (MT)": "weekly_total_arrivals"}[hv.value]
        agg = "sum" if "arrivals" in hv.value else "mean"
        pv = dd.pivot_table(index="state", columns="week_start_date", values=col, aggfunc=agg)
        if "vs state" in hv.value: pv = pv.sub(pv.mean(axis=1), axis=0).div(pv.mean(axis=1), axis=0) * 100
        h = px.imshow(pv, aspect="auto", color_continuous_scale="RdBu_r" if "vs state" in hv.value else "YlOrRd", height=max(380, 24 * len(pv) + 120),
                      labels=dict(x="Week", y="State", color=hv.value), title=f"{crop.value}: state × week — {hv.value}")
        if "vs state" in hv.value: h.update_coloraxes(cmid=0)
        h.update_layout(margin=dict(t=50, b=20)); FIGS["heatmap"] = h
        show(o_heat, h)
    watch([crop, metric, wk, hv], render); render()
    note = callout("Map positions are the district-HQ coordinates used to fetch weather from Open-Meteo (<code>weather_daily_by_mandi.csv</code>). "
                   "The <code>dim_mandi.csv</code> Power BI export uses state-centroid + jitter, so it is <b>not</b> used here.", "info")
    return W.VBox([W.HBox([crop, metric], layout=LAYOUT_ROW), wk, note, o_map, W.HBox([hv]), o_heat])

# ── TAB 4 · Weather & supply drivers ─────────────────────────────────────────────────────
DRIVERS = [c for c in ["arrivals_vs_roll_mean_4", "arrivals_pct_change_1w", "rainfall_mm", "rainfall_lag_1", "rainfall_lag_2", "rainfall_roll_sum_4", "rainy_days",
                       "temp_mean_c", "temp_max_c", "temp_anomaly_c", "humidity_mean_pct", "pct_change_1w", "pct_change_4w", "price_vs_roll_mean_4",
                       "price_vs_state_mean", "price_spread_pct"] if c in feats.columns]
def build_drivers():
    crop = W.Dropdown(options=["All crops"] + CROPS, description="Crop", layout=W.Layout(width="230px"))
    xv = W.Dropdown(options=DRIVERS, value="rainfall_lag_1" if "rainfall_lag_1" in DRIVERS else DRIVERS[0], description="Driver (x)", layout=W.Layout(width="340px"))
    colour = W.Dropdown(options=["target_direction", "crop", "state", "None"], description="Colour", layout=W.Layout(width="260px"))
    nb = W.IntSlider(value=6, min=4, max=12, description="Bins", layout=W.Layout(width="240px"))
    clip = W.Checkbox(value=True, description="Clip outliers (1–99 %)", indent=False)
    flag = W.Dropdown(options=[c for c in ["heavy_rain_flag", "heat_flag"] if c in feats.columns], description="Flag test", layout=W.Layout(width="260px"))
    o1, o2, o3 = out(), out(), out()

    def sub():
        d = feats if crop.value == "All crops" else feats[feats.crop == crop.value]
        return d

    def render():
        d = sub()[[xv.value, "target_pct_change", "target_direction", "crop", "state"]].dropna()
        if clip.value:
            for c in (xv.value, "target_pct_change"):
                lo, hi = d[c].quantile([.01, .99]); d = d[(d[c] >= lo) & (d[c] <= hi)]
        rho, p = stats.spearmanr(d[xv.value], d.target_pct_change)
        slope, icpt = np.polyfit(d[xv.value], d.target_pct_change, 1)
        sample = d.sample(min(len(d), 3500), random_state=1)
        fig = make_subplots(rows=1, cols=2, subplot_titles=("Next-week price change vs driver", "Binned: mean next-week change (±95 % CI)"), column_widths=[.55, .45])
        cmap = {"rise": "#2f9e44", "fall": "#e03131", "flat": "#868e96"}
        if colour.value == "None": fig.add_trace(go.Scattergl(x=sample[xv.value], y=sample.target_pct_change, mode="markers", marker=dict(size=4, opacity=.35, color="#495057"), name="obs"), 1, 1)
        else:
            for k, gp in sample.groupby(colour.value):
                fig.add_trace(go.Scattergl(x=gp[xv.value], y=gp.target_pct_change, mode="markers", name=str(k), marker=dict(size=4, opacity=.4, color=cmap.get(k, COL.get(k))) if colour.value in ("target_direction", "crop") else dict(size=4, opacity=.4)), 1, 1)
        xs = np.linspace(d[xv.value].min(), d[xv.value].max(), 50)
        fig.add_trace(go.Scatter(x=xs, y=slope * xs + icpt, name="linear fit", line=dict(color="black", width=2)), 1, 1)
        q = pd.qcut(d[xv.value], nb.value, duplicates="drop")
        b = d.groupby(q, observed=True).agg(m=("target_pct_change", "mean"), s=("target_pct_change", "sem"), x=(xv.value, "mean"), n=("target_pct_change", "size")).reset_index()
        fig.add_trace(go.Bar(x=[f"{v:.1f}" for v in b.x], y=b.m, error_y=dict(type="data", array=1.96 * b.s), marker_color=["#2f9e44" if v > 0 else "#e03131" for v in b.m], showlegend=False,
                             hovertext=[f"n = {n}" for n in b.n]), 1, 2)
        fig.update_xaxes(title_text=xv.value, row=1, col=1); fig.update_xaxes(title_text=f"{xv.value} (bin mean)", row=1, col=2); fig.update_yaxes(title_text="Next-week change %", row=1, col=1)
        fig.update_layout(height=450, margin=dict(t=50, b=20), legend=dict(orientation="h", y=-0.2)); FIGS["drivers"] = fig
        sig = "statistically detectable" if p < 0.05 else "not statistically distinguishable from zero"
        k = kpis([("Spearman ρ", f"{rho:+.3f}", f"p = {p:.3g}", "inherit"), ("Linear slope", f"{slope:+.4f}", "pp change per unit of x", "inherit"), ("Observations", f"{len(d):,}", "after clipping" if clip.value else "all", "inherit")])
        msg = callout(f"The association between <b>{xv.value}</b> and next-week price change is {sig} (|ρ| = {abs(rho):.2f}). "
                      "Panel rows from the same mandi and week are correlated, so p-values are optimistic; read ρ as a screening statistic, not proof.", "good" if p < .05 and abs(rho) >= .1 else "info")
        show(o1, k, msg, fig)
        # lagged-correlation heatmap
        rows = {}
        for cn in ["All crops"] + CROPS:
            dd = feats if cn == "All crops" else feats[feats.crop == cn]
            rows[cn] = [stats.spearmanr(dd[c], dd.target_pct_change, nan_policy="omit")[0] for c in DRIVERS]
        cm = pd.DataFrame(rows, index=DRIVERS).T
        h = px.imshow(cm, color_continuous_scale="RdBu_r", zmin=-.4, zmax=.4, aspect="auto", text_auto=".2f", height=290, title="Spearman ρ with next-week price change (all drivers)")
        h.update_layout(margin=dict(t=50, b=10)); FIGS["driver_corr"] = h
        show(o2, h)
        # flag test
        f = sub(); rows = []
        for cn in CROPS:
            for v in (0, 1):
                s = f[(f.crop == cn) & (f[flag.value] == v)].target_pct_change
                rows.append(dict(crop=cn, flag=bool(v), mean=s.mean(), ci=1.96 * s.sem(), n=len(s)))
        fd = pd.DataFrame(rows)
        fg = px.bar(fd, x="crop", y="mean", color="flag", barmode="group", error_y="ci", color_discrete_map={False: "#adb5bd", True: "#1c7ed6"}, height=330,
                    title=f"Mean next-week price change when {flag.value} is on vs off", labels={"mean": "Next-week change %"})
        pvals = []
        for cn in CROPS:
            a = f[(f.crop == cn) & (f[flag.value] == 1)].target_pct_change; z = f[(f.crop == cn) & (f[flag.value] == 0)].target_pct_change
            if len(a) > 2 and len(z) > 2: pvals.append(f"{cn}: Welch p = {stats.ttest_ind(a, z, equal_var=False).pvalue:.3f}")
        fg.update_layout(margin=dict(t=50, b=10)); show(o3, fg, callout(" · ".join(pvals) + " (t-test on next-week % change, flag on vs off)", "info"))
    watch([crop, xv, colour, nb, clip, flag], render); render()
    return W.VBox([W.HBox([crop, xv, colour], layout=LAYOUT_ROW), W.HBox([nb, clip], layout=LAYOUT_ROW), o1, o2, W.HBox([flag]), o3])

In [ ]:
# ── TAB 5 · Forecast Lab ─────────────────────────────────────────────────────────────────
def build_forecast():
    crop = W.ToggleButtons(options=CROPS, description="Crop", style={"button_width": "80px"})
    hz = W.ToggleButtons(options=[1, 2, 3, 4], description="Horizon (wk)", style={"button_width": "50px"})
    models = W.SelectMultiple(options=[(MODEL_LBL[m], m) for m in MODEL_COL], value=("Naive", "Seasonal_Naive", "SARIMA_Univariate"), description="Models", rows=5, layout=W.Layout(width="340px"))
    hist = W.IntSlider(value=26, min=8, max=N_WEEKS, description="History (wk)", layout=W.Layout(width="320px"))
    o_cmp, o_tab, o_fan = out(), out(), out()

    def render():
        c, h = crop.value, hz.value
        d = bt[(bt.Crop == c) & (bt.Horizon == h)]
        fig = go.Figure(); a = d.drop_duplicates("Forecast_Date").sort_values("Forecast_Date")
        fig.add_trace(go.Scatter(x=a.Forecast_Date, y=a.Actual, name="Actual", mode="lines+markers", line=dict(color="black", width=3)))
        for m in models.value:
            dm = d[d.Model == m].sort_values("Forecast_Date")
            fig.add_trace(go.Scatter(x=dm.Forecast_Date, y=dm.Predicted, name=MODEL_LBL[m], mode="lines+markers", line=dict(color=MODEL_COL[m], width=2, dash="dot" if m == "Naive" else "solid")))
        fig.update_layout(height=400, title=f"{c}: backtest, {h}-week-ahead forecasts vs actual (8 rolling origins)", yaxis_title="₹ / quintal", hovermode="x unified", margin=dict(t=50, b=20), legend=dict(orientation="h", y=-0.15))
        FIGS["forecast_backtest"] = fig
        w = piv[(piv.Crop == c) & (piv.Horizon == h)]; k = int(w.win_vs_snaive.sum()); kn = int(w.win_vs_naive.sum()); n = len(w)
        msg = callout(f"At {h} week(s) for {c}, SARIMA had a smaller absolute error than <b>seasonal naive in {k} of {n}</b> folds and than <b>naive in {kn} of {n}</b>. "
                      + ("A genuine edge over seasonal naive is not demonstrated with so few folds." if k < n - 1 else "Consistent advantage across folds, but still only 8 origins."), "info" if k >= n / 2 else "warn")
        show(o_cmp, msg, fig)
        # metrics table + grouped bars
        t = mape_tbl[mape_tbl.Crop == c].merge(sar_eval[["Crop", "Model", "Horizon", "RMSE", "MASE"]], on=["Crop", "Model", "Horizon"], how="left")
        t["Model"] = t.Model.map(MODEL_LBL); t = t[["Model", "Horizon", "MAE", "RMSE", "MAPE", "MASE", "Folds"]].sort_values(["Horizon", "MAPE"])
        gb = px.bar(mape_tbl[mape_tbl.Crop == c].assign(Model=lambda x: x.Model.map(MODEL_LBL)), x="Horizon", y="MAPE", color="Model", barmode="group", log_y=True, height=340,
                    color_discrete_map={MODEL_LBL[m]: MODEL_COL[m] for m in MODEL_COL}, title=f"{c}: MAPE % by model and horizon (log scale)")
        gb.update_layout(margin=dict(t=50, b=10), xaxis=dict(dtick=1)); FIGS["forecast_mape"] = gb; STATE["forecast_metrics"] = t
        show(o_tab, gb, table(t, fmt={"MAE": "{:.1f}", "RMSE": "{:.1f}", "MAPE": "{:.2f}", "MASE": "{:.2f}"}, gradient=["MAPE"]))
        # future fan
        f = fut[fut.Crop == c].sort_values("Week_Start_Date"); hh = nat[nat.crop == c].tail(hist.value)
        fan = go.Figure()
        fan.add_trace(go.Scatter(x=hh.week_start_date, y=hh.price, name="History (national mean)", line=dict(color="black", width=2.4)))
        fan.add_trace(go.Scatter(x=list(f.Week_Start_Date) + list(f.Week_Start_Date[::-1]), y=list(f.Upper_95) + list(f.Lower_95[::-1]), fill="toself", fillcolor="rgba(47,158,68,.12)", line=dict(width=0), name="95 % interval"))
        fan.add_trace(go.Scatter(x=list(f.Week_Start_Date) + list(f.Week_Start_Date[::-1]), y=list(f.Upper_80) + list(f.Lower_80[::-1]), fill="toself", fillcolor="rgba(47,158,68,.28)", line=dict(width=0), name="80 % interval"))
        fan.add_trace(go.Scatter(x=f.Week_Start_Date, y=f.Forecast_Price, name="SARIMA forecast", mode="lines+markers", line=dict(color="#2f9e44", width=3)))
        fan.update_layout(height=400, title=f"{c}: 4-week-ahead SARIMA forecast with 80 / 95 % intervals", yaxis_title="₹ / quintal", hovermode="x unified", margin=dict(t=50, b=20), legend=dict(orientation="h", y=-0.15))
        FIGS["forecast_fan"] = fan; STATE["forecast_future"] = f
        last = hh.price.iloc[-1]
        show(o_fan, fan, callout(f"Intervals come from the model's in-sample error structure; their out-of-sample coverage was <b>not</b> tested (only 8 folds). "
                                 f"Last observed national price: {inr(last)}; 4-week point forecast: {inr(f.Forecast_Price.iloc[-1])} ({(f.Forecast_Price.iloc[-1]/last-1)*100:+.1f}%).", "warn"))
    watch([crop, hz, models, hist], render); render()
    return W.VBox([W.HBox([crop, hz], layout=LAYOUT_ROW), W.HBox([models, hist], layout=LAYOUT_ROW), o_cmp, o_tab, o_fan])

In [ ]:
# ── TAB 6 · Direction models ─────────────────────────────────────────────────────────────
def build_models():
    m = mres.copy(); m["kind"] = np.where(m.experiment.str.contains("Baselines"), "Baseline", "Model")
    m["label"] = m.baseline
    sel = W.SelectMultiple(options=list(m.label), value=tuple(m.label), description="Models", rows=9, layout=W.Layout(width="360px"))
    ho_metric = W.Dropdown(options=["Macro_F1", "Accuracy", "Weighted_F1", "ROC_AUC_OvR", "Recall_Rise", "Recall_Fall", "Recall_Flat"], description="Hold-out metric", layout=W.Layout(width="300px"))
    sp_metric = W.Dropdown(options=["Binary_F1", "Precision", "Recall", "ROC_AUC", "Binary_Accuracy"], description="Spike metric", layout=W.Layout(width="300px"))
    figs_dir = [REP / "figures" / s for s in ("task6", "deep_learning", "task4")]
    gallery = sorted([p for d in figs_dir if d.exists() for p in d.glob("*.png")])
    gal = W.Dropdown(options=[(f"{p.parent.name} / {p.stem}", str(p)) for p in gallery], description="Saved figure", layout=W.Layout(width="520px")) if gallery else None
    o1, o2, o3, og = out(), out(), out(), out()  # o4 removed (was unused)

    def render():
        d = m[m.label.isin(sel.value)].sort_values("mean_macro_f1")
        f1 = go.Figure(go.Bar(y=d.label, x=d.mean_macro_f1, orientation="h", error_x=dict(type="data", array=d.std_macro_f1), marker_color=["#adb5bd" if k == "Baseline" else "#2f9e44" for k in d.kind],
                              text=[f"{v:.3f}" for v in d.mean_macro_f1], textposition="outside"))
        f1.update_layout(height=max(300, 34 * len(d) + 90), title="Time-series CV: macro-F1 (mean ± std across folds) — grey = naive baselines", xaxis_title="Macro-F1", margin=dict(t=50, b=20), xaxis_range=[0, max(.75, d.mean_macro_f1.max() + d.std_macro_f1.max() + .08)])
        rec = d.melt(id_vars="label", value_vars=["mean_fall_recall", "mean_flat_recall", "mean_rise_recall"], var_name="class", value_name="recall")
        rec["class"] = rec["class"].str.replace("mean_", "").str.replace("_recall", "")
        f2 = px.bar(rec, x="label", y="recall", color="class", barmode="group", height=340, color_discrete_map={"fall": "#e03131", "flat": "#868e96", "rise": "#2f9e44"}, title="Per-class recall (does the model catch each direction?)", labels={"label": ""})
        f2.update_layout(margin=dict(t=50, b=10)); FIGS["models_cv"] = f1; FIGS["models_recall"] = f2
        ml = m[m.kind == "Model"]; top = ml.sort_values("mean_macro_f1", ascending=False).iloc[0]
        close = ml[(top.mean_macro_f1 - ml.mean_macro_f1) <= top.std_macro_f1]
        msg = callout(f"<b>{len(close)} of {len(ml)}</b> trained models fall within one standard deviation of the best ({top.label}, {top.mean_macro_f1:.3f} ± {top.std_macro_f1:.3f}). "
                      "Differences between them are smaller than fold-to-fold noise; all clearly beat the naive baselines. "
                      "<b>Flat</b> is the hardest class (recall ≈ 0.35–0.43) while rise / fall are caught far more often.", "warn")
        show(o1, msg, f1, f2)
        h = dlml.sort_values(ho_metric.value); hf = px.bar(h, y="Model", x=ho_metric.value, orientation="h", height=max(300, 36 * len(h) + 90), color=ho_metric.value, color_continuous_scale="Viridis",
                                                          title=f"Hold-out evaluation (±5 % band): {ho_metric.value}", text=h[ho_metric.value].round(3))
        hf.update_layout(margin=dict(t=50, b=10), coloraxis_showscale=False); FIGS["models_holdout"] = hf
        note = callout("Tuned XGBoost / CatBoost show the highest <i>accuracy</i> on this hold-out but a lower <i>macro-F1</i> than logistic regression and random forest: they lean toward the majority class. "
                       "Deep SimpleRNN / LSTM models underperform tree models here (small sample). Pick the metric that matches the decision.", "info")
        show(o2, note, hf)
        b = bin5.sort_values(sp_metric.value); bf = px.bar(b, y="Model", x=sp_metric.value, orientation="h", height=max(300, 36 * len(b) + 90), color=sp_metric.value, color_continuous_scale="Tealgrn",
                                                          title=f"Binary 'price rises >5 %' detector: {sp_metric.value}", text=b[sp_metric.value].round(3))
        bf.update_layout(margin=dict(t=50, b=10), coloraxis_showscale=False); show(o3, bf)
    def render_gal():
        if gal is not None: show(og, Image(filename=gal.value, width=900))
    watch([sel, ho_metric, sp_metric], render); render()
    kids = [W.HBox([sel, W.VBox([ho_metric, sp_metric])], layout=LAYOUT_ROW), o1, o2, o3]
    if gal is not None: watch([gal], render_gal); render_gal(); kids += [HTML("<h4>Saved figures from the modelling notebooks</h4>"), gal, og]
    return W.VBox(kids)

# ── TAB 7 · Rules explorer ───────────────────────────────────────────────────────────────
def rule_card(rid):
    r = rules[rules.Rule_ID == rid].iloc[0]
    flags = []
    if r.trivial: flags.append("<span class='ap-badge' style='background:#e03131'>contains an always-true item</span>")
    if r.redundant: flags.append("<span class='ap-badge' style='background:#f08c00'>redundant (a simpler rule is as confident)</span>")
    html = (f"<div class='ap-call ap-info'><b>{r.Rule_ID}</b> · {r.Crop} · {r.Target_Description} {' '.join(flags)}<br>{r.Rule_Statement}<br>"
            f"<b>Trigger:</b> {r.Trigger_Condition}<br><b>Suggested action:</b> {r.Recommended_Action}<br>"
            f"<b>Support</b> {r.Support*100:.2f}% · <b>Confidence</b> {r.Confidence*100:.1f}% · <b>Lift</b> {r.Lift:.2f}× · <b>Conviction</b> {r.Conviction:.2f} · "
            f"<b>Base rate of outcome</b> {r.baseline_p*100:.1f}%<br><i>Association, not causation.</i></div>")
    c = rcv[rcv.Rule_ID == rid]
    if c.empty:
        return [HTML(html), callout("No cross-crop test stored for this rule (the cross-validation file covers the top 20 rules only).", "warn")]
    fig = go.Figure(go.Bar(x=c.Test_Crop, y=c.Empirical_Confidence * 100, marker_color=[COL[t] for t in c.Test_Crop], text=[f"{v*100:.1f}%  (n={n})" for v, n in zip(c.Empirical_Confidence, c.Trigger_Count)], textposition="outside"))
    fig.add_hline(y=c.Global_Confidence.iloc[0] * 100, line_dash="dash", line_color="black", annotation_text="all-crop confidence")
    fig.add_hline(y=c.baseline_p.iloc[0] * 100, line_dash="dot", line_color="#868e96", annotation_text="base rate")
    fig.update_layout(height=300, title="Does the rule hold within each crop?", yaxis_title="Confidence %", margin=dict(t=50, b=10), yaxis_range=[0, 100])
    holds = int(c.holds.sum())
    return [HTML(html), fig, callout(f"Within-crop confidence beats the base rate in <b>{holds} of {len(c)}</b> crops, but is {abs(c.gap_pts.mean()):.0f} points {'lower' if c.gap_pts.mean() < 0 else 'higher'} than the pooled figure on average.", "good" if holds == len(c) and abs(c.gap_pts.mean()) < 10 else "warn")]

def build_rules():
    crop = W.Dropdown(options=["All"] + sorted(rules.Crop.unique()), description="Crop", layout=W.Layout(width="230px"))
    outc = W.Dropdown(options=["All"] + sorted(rules.Target_Outcome.unique()), description="Outcome", layout=W.Layout(width="330px"))
    ms = W.FloatSlider(value=rules.Support.min(), min=rules.Support.min(), max=.2, step=.005, description="Min support", readout_format=".3f", layout=W.Layout(width="330px"))
    mc = W.FloatSlider(value=.5, min=.49, max=.77, step=.01, description="Min confidence", readout_format=".2f", layout=W.Layout(width="330px"))
    ml = W.FloatSlider(value=1.05, min=1.05, max=1.46, step=.01, description="Min lift", readout_format=".2f", layout=W.Layout(width="330px"))
    sortby = W.Dropdown(options=["Lift", "Confidence", "Support", "Conviction"], description="Sort by", layout=W.Layout(width="230px"))
    topn = W.IntSlider(value=15, min=5, max=40, description="Top N", layout=W.Layout(width="260px"))
    q = W.Text(placeholder="e.g. Heavy_Rain or Onion", description="Contains", layout=W.Layout(width="330px"))
    ht = W.Checkbox(value=True, description="Hide trivial", indent=False); hr = W.Checkbox(value=True, description="Hide redundant", indent=False)
    pick = W.Dropdown(options=[], description="Rule", layout=W.Layout(width="760px"))
    o_k, o_s, o_b, o_t, o_c = out(), out(), out(), out(), out()

    def filt():
        d = rules[(rules.Support >= ms.value) & (rules.Confidence >= mc.value) & (rules.Lift >= ml.value)]
        if crop.value != "All": d = d[d.Crop == crop.value]
        if outc.value != "All": d = d[d.Target_Outcome == outc.value]
        if ht.value: d = d[~d.trivial]
        if hr.value: d = d[~d.redundant]
        if q.value.strip(): d = d[d.Antecedents.str.contains(q.value.strip(), case=False, regex=False) | d.Rule_Statement.str.contains(q.value.strip(), case=False, regex=False)]
        return d.sort_values(sortby.value, ascending=False)

    def render(*_):
        d = filt(); STATE["rules_filtered"] = d.drop(columns=["items"])
        if d.empty:
            show(o_k, callout("No rules match these filters. Loosen a threshold.", "warn")); show(o_s); show(o_b); show(o_t); show(o_c); return
        show(o_k, kpis([("Rules shown", f"{len(d)}", f"of {len(rules)}", "#7048e8"), ("Mean confidence", pct(d.Confidence.mean() * 100), f"max {pct(d.Confidence.max() * 100)}", "inherit"),
                        ("Mean lift", f"{d.Lift.mean():.2f}×", f"max {d.Lift.max():.2f}×", "inherit"), ("Weather-based", f"{int(d.weather_rule.sum())}", "use rain / heat / humidity", "inherit")]))
        sc = px.scatter(d, x="Support", y="Confidence", color="Lift", size="n_items", symbol="Target_Outcome", hover_name="Rule_ID", hover_data=["Crop", "Trigger_Condition"],
                        color_continuous_scale="Viridis", height=430, title="Support vs confidence (colour = lift, size = number of conditions)")
        sc.update_layout(margin=dict(t=50, b=10)); FIGS["rules_scatter"] = sc; show(o_s, sc)
        t = d.head(topn.value).iloc[::-1]
        bar = go.Figure(go.Bar(y=t.Rule_ID + " · " + t.Crop, x=t[sortby.value], orientation="h", marker_color=[{"Price_Spike_NextWeek": "#e8590c", "Severe_Spike_NextWeek": "#c92a2a", "Price_Crash_NextWeek": "#1c7ed6", "Severe_Crash_NextWeek": "#364fc7"}[o] for o in t.Target_Outcome],
                               text=t.Antecedents.str.replace("_", " ").str.slice(0, 80), textposition="inside", insidetextanchor="start", hovertext=t.Rule_Statement))
        bar.update_layout(height=max(320, 28 * len(t) + 90), title=f"Top {len(t)} by {sortby.value.lower()} (orange = spike rules, blue = crash rules)", margin=dict(t=50, b=10), xaxis_title=sortby.value); show(o_b, bar)
        cols = ["Rule_ID", "Crop", "Target_Outcome", "Antecedents", "Support", "Confidence", "Lift", "Conviction", "trivial", "redundant"]
        show(o_t, table(d[cols].head(60), fmt={"Support": "{:.3f}", "Confidence": "{:.3f}", "Lift": "{:.3f}", "Conviction": "{:.2f}"}, gradient=None))
        opts = [(f"{r.Rule_ID} · {r.Crop} · conf {r.Confidence*100:.0f}% · lift {r.Lift:.2f} · {r.Antecedents.replace('_', ' ')[:70]}", r.Rule_ID) for r in d.head(60).itertuples()]
        keep = pick.value if pick.value in [v for _, v in opts] else opts[0][1]
        def _pick_render(change):
            if change["name"] == "value": render_card()
        pick.unobserve_all("value"); pick.options = opts; pick.value = keep; pick.observe(_pick_render, names="value"); render_card()
    def render_card():
        items = rule_card(pick.value)
        show(o_c, *items)
    watch([crop, outc, ms, mc, ml, sortby, topn, q, ht, hr], render); render()
    note = callout(f"<b>Quality filters.</b> <code>High_Price_Spread</code> (spread &gt; 15 %) is true for {SPREAD_SHARE*100:.0f}% of mandi-weeks, so it adds no information; "
                   f"{int(rules.trivial.sum())} rules contain it. A rule is flagged <i>redundant</i> when a simpler rule (subset of conditions) for the same crop and outcome is at least as confident.", "info")
    return W.VBox([W.HBox([crop, outc, sortby], layout=LAYOUT_ROW), W.HBox([ms, mc, ml], layout=LAYOUT_ROW), W.HBox([q, topn, ht, hr], layout=LAYOUT_ROW), note, o_k, o_s, o_b, HTML("<h4>Rule table</h4>"), o_t, HTML("<h4>Inspect a rule</h4>"), pick, o_c])

# ── TAB 8 · Spike-risk scorer ────────────────────────────────────────────────────────────
GROUPS = {"Weather": [i for i in ALL_ITEMS if re.search(r"Rain|Heat|Humid", i)],
          "Supply (arrivals)": [i for i in ALL_ITEMS if re.search(r"Arrivals|Supply", i)],
          "Price state": [i for i in ALL_ITEMS if re.search(r"Momentum|Spread", i)]}
def build_scorer():
    pretty = lambda s: s.replace("_", " ")
    cb = {i: W.Checkbox(value=False, description=pretty(i), indent=False, layout=W.Layout(width="330px")) for g in GROUPS.values() for i in g}
    crop = W.Dropdown(options=["Not specified"] + CROPS, description="Crop", layout=W.Layout(width="260px"))
    reg = W.Dropdown(options=["Not specified", "North", "South", "West"], description="Region", layout=W.Layout(width="260px"))
    outc = W.Dropdown(options=sorted(rules.Target_Outcome.unique()), value="Price_Spike_NextWeek", description="Outcome", layout=W.Layout(width="340px"))
    b1 = W.Button(description="Preset: rain squeeze", button_style="warning"); b2 = W.Button(description="Preset: humid glut (crash risk)", button_style="info"); b3 = W.Button(description="Clear", button_style="")
    o = out()
    def preset(items):
        for k, w in cb.items(): w.value = k in items
    b1.on_click(lambda _: preset({"Heavy_Rain_Lag1", "Low_Arrivals_CurrentWeek", "Bearish_Price_Momentum"}))
    b2.on_click(lambda _: (preset({"High_Humidity_CurrentWeek", "Arrivals_Surge_CurrentWeek", "Bullish_Price_Momentum"}), setattr(outc, "value", "Price_Crash_NextWeek")))
    b3.on_click(lambda _: preset(set()))
    def render(*_):
        obs = {k for k, w in cb.items() if w.value} | TRIVIAL_ITEMS
        if crop.value != "Not specified": obs.add(f"Crop_{crop.value}")
        if reg.value != "Not specified": obs.add(f"Region_{reg.value}")
        m = rules[(rules.Target_Outcome == outc.value) & rules["items"].apply(lambda s: s <= obs)].sort_values("Confidence", ascending=False)
        STATE["scorer_matches"] = m.drop(columns=["items"])
        if m.empty:
            show(o, callout("No mined rule fires for exactly these conditions. That is itself informative: this combination was not frequent enough (support ≥ 2.5 %) "
                            "or did not raise the odds enough to be kept. Try ticking the conditions that apply together (e.g. last week's heavy rain <i>and</i> low arrivals).", "warn")); return
        best = m.iloc[0]
        g = go.Figure(go.Indicator(mode="gauge+number+delta", value=best.Confidence * 100, number={"suffix": "%"}, delta={"reference": best.baseline_p * 100, "suffix": " pts vs base rate", "increasing": {"color": "#e8590c"}},
                                   title={"text": f"Best matching rule: {best.Rule_ID}<br><span style='font-size:12px'>{best.Target_Description}</span>"},
                                   gauge={"axis": {"range": [0, 100]}, "bar": {"color": "#e8590c"}, "threshold": {"line": {"color": "black", "width": 3}, "value": best.baseline_p * 100}}))
        g.update_layout(height=290, margin=dict(t=90, b=10)); FIGS["scorer"] = g
        lst = m.head(12)[["Rule_ID", "Crop", "Antecedents", "Support", "Confidence", "Lift"]]
        show(o, g, callout(f"{len(m)} rule(s) fire. Highest confidence <b>{best.Confidence*100:.1f}%</b> against a base rate of <b>{best.baseline_p*100:.1f}%</b> (lift {best.Lift:.2f}×). "
                           "Rules overlap, so the numbers are <b>not</b> additive: read the highest one, not the sum. Confidence is a historical frequency over 53 weeks, not a calibrated probability.", "info"),
             table(lst, fmt={"Support": "{:.3f}", "Confidence": "{:.3f}", "Lift": "{:.3f}"}))
    watch([crop, reg, outc, *cb.values()], render); render()
    cols = [W.VBox([HTML(f"<b>{g}</b>")] + [cb[i] for i in items]) for g, items in GROUPS.items()]
    note = callout("Tick the conditions you observe <i>this week</i> (or last week for the 'Lag1' items). <code>High Price Spread</code> is always counted as true, because it is true in every week of the data.", "info")
    return W.VBox([note, W.HBox([crop, reg, outc], layout=LAYOUT_ROW), W.HBox(cols, layout=LAYOUT_ROW), W.HBox([b1, b2, b3]), o])

# ── TAB 9 · Procurement planner ──────────────────────────────────────────────────────────
def build_planner():
    crop = W.ToggleButtons(options=CROPS, description="Crop", style={"button_width": "80px"})
    last = lambda c: float(nat[nat.crop == c].price.iloc[-1])
    p0 = W.BoundedFloatText(value=round(last("Tomato")), min=100, max=20000, step=10, description="Price now ₹/qtl", layout=W.Layout(width="260px"))
    qty = W.IntSlider(value=500, min=50, max=5000, step=50, description="Quintals / wk", layout=W.Layout(width="340px"))
    shock = W.IntSlider(value=0, min=-10, max=20, description="Stress shock %", layout=W.Layout(width="320px"))
    trig = W.Dropdown(options=["No rule trigger"], description="If rule fires", layout=W.Layout(width="620px"))
    o = out()
    def set_defaults(*_):
        trig.unobserve_all("value")  # reset before re-wiring to avoid duplicate observers
        p0.value = round(last(crop.value))
        d = rules[(rules.Target_Outcome == "Price_Spike_NextWeek") & ~rules.trivial & ~rules.redundant & rules.Crop.isin(["Any Crop", crop.value])].sort_values("Confidence", ascending=False).head(8)
        trig.unobserve_all("value"); trig.options = [("No rule trigger", None)] + [(f"{r.Rule_ID} · conf {r.Confidence*100:.0f}% · {r.Antecedents.replace('_', ' ')[:70]}", r.Rule_ID) for r in d.itertuples()]; trig.value = None
        trig.observe(lambda c: render(), names="value")
    def render(*_):
        f = fut[fut.Crop == crop.value].sort_values("Horizon_Weeks_Ahead").copy(); P0 = p0.value; q = qty.value; s = 1 + shock.value / 100
        for c in ["Forecast_Price", "Lower_80", "Upper_80"]: f[c] = f[c] * s
        f["delta"] = (f.Forecast_Price - P0) * q / 1e5; f["lo"] = (f.Lower_80 - P0) * q / 1e5; f["hi"] = (f.Upper_80 - P0) * q / 1e5
        def verdict(r): return "Buy now: waiting likely costs more" if r.Lower_80 > P0 else ("Waiting likely cheaper" if r.Upper_80 < P0 else "Inconclusive (interval spans today's price)")
        f["Reading (80 % interval)"] = f.apply(verdict, axis=1)
        fig = go.Figure(go.Bar(x=[f"Week +{h}" for h in f.Horizon_Weeks_Ahead], y=f.delta, marker_color=["#e03131" if v > 0 else "#2f9e44" for v in f.delta],
                               error_y=dict(type="data", symmetric=False, array=f.hi - f.delta, arrayminus=f.delta - f.lo), text=[f"{v:+.2f}" for v in f.delta], textposition="outside"))
        fig.add_hline(y=0, line_color="black"); fig.update_layout(height=380, margin=dict(t=55, b=10), yaxis_title="₹ lakh vs buying today", title=f"{crop.value}: extra cost (+) or saving (−) of buying {q:,} qtl later instead of today (bars = point forecast, whiskers = 80 % interval)")
        FIGS["planner"] = fig; STATE["planner_table"] = f
        extra = []
        if trig.value:
            r = rules[rules.Rule_ID == trig.value].iloc[0]
            extra = [callout(f"If <b>{r.Rule_ID}</b> is active, history says the chance of a >5 % rise next week moves from <b>{r.baseline_p*100:.0f}%</b> (base rate) to <b>{r.Confidence*100:.0f}%</b>. "
                             f"A +5 % move on {q:,} qtl at ₹{P0:,.0f} ≈ <b>₹{q*P0*0.05/1e5:.2f} lakh</b> extra cost. Use as a prompt to check local conditions, not as a guarantee.", "warn")]
        t = f[["Horizon_Weeks_Ahead", "Week_Start_Date", "Forecast_Price", "Lower_80", "Upper_80", "delta", "Reading (80 % interval)"]].rename(columns={"delta": "Δ cost ₹ lakh"})
        show(o, fig, *extra, table(t, fmt={"Forecast_Price": "{:,.0f}", "Lower_80": "{:,.0f}", "Upper_80": "{:,.0f}", "Δ cost ₹ lakh": "{:+.2f}", "Week_Start_Date": "{:%d %b %Y}"}),
             callout("Illustrative calculator. Forecast intervals are narrow relative to real price risk (model fitted on one year of national averages), and no cost-saving backtest has been run. "
                     "Earlier report figures such as '8–12 % procurement savings' are <b>not</b> reproduced here because no analysis in the repo supports them.", "bad"))
    crop.observe(lambda c: (set_defaults(), render()), names="value"); watch([p0, qty, shock], render); set_defaults(); render()
    return W.VBox([W.HBox([crop, p0], layout=LAYOUT_ROW), W.HBox([qty, shock], layout=LAYOUT_ROW), trig, o])

In [ ]:
# ── TAB 10 · Data & methods QA ───────────────────────────────────────────────────────────
def hav(lat1, lon1, lat2, lon2):
    p = np.pi / 180; a = np.sin((lat2 - lat1) * p / 2) ** 2 + np.cos(lat1 * p) * np.cos(lat2 * p) * np.sin((lon2 - lon1) * p / 2) ** 2
    return 12742 * np.arcsin(np.sqrt(a))

def build_qa():
    crop = W.ToggleButtons(options=CROPS, description="Crop", style={"button_width": "80px"})
    o_top, o_cov, o_stat = out(), out(), out()
    per = panel.groupby(["market_name", "crop"]).week_start_date.count()
    bad_range = int(((panel.weekly_min_price > panel.weekly_modal_price) | (panel.weekly_modal_price > panel.weekly_max_price)).sum())
    part = panel[panel.is_partial_week].week_start_date.dt.strftime("%d %b %Y").unique()
    k = kpis([("Series", f"{per.size}", "mandi × crop", "inherit"), ("Weeks per series", f"{per.min()}–{per.max()}", "balanced panel" if per.min() == per.max() else "unbalanced", "inherit"),
              ("Missing values", f"{int(panel.isna().sum().sum())}", "in the weekly panel", "inherit"), ("Partial weeks", f"{int(panel.is_partial_week.sum())}", ", ".join(part[:2]), "inherit"),
              ("Min > modal or modal > max", f"{bad_range}", "price-range inconsistencies", "inherit"), ("Non-positive prices", f"{int((panel.weekly_modal_price <= 0).sum())}", "", "inherit")])
    msgs = []
    if dim_mandi_path.exists():
        dm = pd.read_csv(dim_mandi_path).merge(coords, left_on="Market_Name", right_on="market_name")
        km = hav(dm.Latitude, dm.Longitude, dm.latitude, dm.longitude)
        msgs.append(callout(f"<b>Coordinate check.</b> The Power BI file <code>dim_mandi.csv</code> places mandis a median <b>{np.median(km):.0f} km</b> (max {km.max():.0f} km) from the coordinates used to fetch weather. "
                            "It is built from state centroids plus offsets, with heuristic region / tier labels, so do not use it for geography or distance-based features.", "bad"))
    n_fold = int(bt.Origin_Week.nunique())
    msgs.append(callout(f"<b>Seasonality.</b> The SARIMA models use a 13-week seasonal period estimated from {N_WEEKS} weeks (about one year) and are validated on {n_fold} origins. "
                        "That cannot separate a stable seasonal cycle from a one-off pattern; a second year of data is the most valuable improvement.", "warn"))
    vf = vif.sort_values("VIF", ascending=False).head(15).iloc[::-1]
    vfig = px.bar(vf, x="VIF", y="feature", color="group", orientation="h", log_x=True, height=420, title="Top 15 variance-inflation factors (log scale; VIF > 10 = strong collinearity)")
    vfig.add_vline(x=10, line_dash="dash", line_color="red"); vfig.update_layout(margin=dict(t=50, b=10))
    show(o_top, k, *msgs)
    def render():
        d = panel[panel.crop == crop.value].pivot_table(index="market_name", columns="week_start_date", values="trading_days_active")
        d = d.loc[d.mean(axis=1).sort_values().index]
        h = px.imshow(d, aspect="auto", color_continuous_scale="Blues", height=max(420, 14 * len(d) + 120), labels=dict(color="Trading days"), title=f"{crop.value}: trading days with data per mandi-week (low = thin data)")
        h.update_layout(margin=dict(t=50, b=10)); FIGS["qa_cov"] = h
        st = stat_tbl.copy(); rs = resid_tbl.copy()
        show(o_cov, h)
        show(o_stat, HTML("<h4>Stationarity tests (national weekly series)</h4>"), table(st, fmt={c: "{:.3f}" for c in st.select_dtypes("number").columns}),
             callout("ADF rejects a unit root and KPSS does not reject stationarity for all three crops, so d = 0 is used. With 53 points these tests have low power, so treat as supportive rather than conclusive.", "info"),
             HTML("<h4>SARIMA residual diagnostics</h4>"), table(rs, fmt={c: "{:.3f}" for c in rs.select_dtypes("number").columns}),
             callout("Ljung-Box shows no left-over autocorrelation (white-noise residuals), but Jarque-Bera rejects normality: shocks are heavy-tailed, so the normal-theory 80 / 95 % intervals are probably too narrow in spike weeks.", "warn"), vfig)
    watch([crop], render); render()
    return W.VBox([o_top, W.HBox([crop]), o_cov, o_stat])

# ── TAB 11 · Literature gaps ─────────────────────────────────────────────────────────────
def build_gaps():
    tom = panel[panel.crop == "Tomato"].groupby("market_name").weekly_modal_price.mean()
    wr, nwr = rules[rules.weather_rule].Lift.median(), rules[~rules.weather_rule].Lift.median()
    ml = mres[~mres.experiment.str.contains("Baselines")].sort_values("mean_macro_f1", ascending=False).iloc[0]
    base = mres[mres.experiment.str.contains("Baselines")].mean_macro_f1.max()
    sx, un = mean_mape["SARIMAX_Weather"], mean_mape["SARIMA_Univariate"]
    fig_exists = (REP / "figures" / "task6" / "feature_importance_top20.png").exists()
    GAPS = [
        ("Addressed", "Single-series or single-market studies ignore cross-mandi heterogeneity",
         f"Market × crop × week panel of {N_MANDI} mandis in {N_STATE} states, built from self-collected CEDA / Agmarknet and Open-Meteo data.",
         f"Tomato mean price ranges {inr(tom.min())}–{inr(tom.max())} across mandis ({tom.max()/tom.min():.2f}×), so a single national series hides large spatial spread.",
         "Forecasts are still made on the national mean only (see gap 10)."),
        ("Addressed", "Weather enters models only as linear regressors; non-linear, event-type effects are under-explored",
         "Weather and arrivals are encoded as thresholded events (heavy rain, heat spike, humidity, arrivals drop) and mined with FP-Growth.",
         f"{int(rules.weather_rule.sum())} of {len(rules)} rules use weather events (median lift {wr:.2f}× vs {nwr:.2f}× for others). The linear SARIMAX with weather averaged {sx:.1f}% MAPE vs {un:.2f}% for SARIMA.",
         "Thresholds are fixed (35 mm, 38 °C, 75 %); sensitivity to these cut-offs is untested."),
        ("Partial", "Forecast studies often omit proper baselines and uncertainty intervals",
         "Rolling-origin backtest against naive and seasonal-naive baselines; MAPE, MASE; 80 / 95 % forecast intervals; residual tests.",
         f"SARIMA had the smaller error than seasonal naive in {piv.win_vs_snaive.mean()*100:.0f}% of {len(piv)} comparisons (8 origins).",
         "Interval coverage was not checked out-of-sample; 8 folds are too few for significance tests."),
        ("Partial", "Direction (rise / fall / flat) prediction is rarely benchmarked against persistence with time-aware validation",
         "Chronological split and expanding-window CV; logistic regression, RF, XGBoost, LightGBM, CatBoost, RNN / LSTM versus three naive baselines.",
         f"Best macro-F1 {ml.mean_macro_f1:.2f} ± {ml.std_macro_f1:.2f} ({ml.baseline}) vs {base:.2f} for the best naive baseline; flat class recall stays low.",
         "Top models are statistically indistinguishable; no hold-out beyond the CV period."),
        ("Partial", "Black-box models offer little guidance to practitioners",
         "Association rules written as plain-language triggers with actions; feature-importance and confusion-matrix figures saved.",
         f"{len(rules)} rules with statements; feature-importance figure {'present' if fig_exists else 'missing'}; {int((~rules.trivial & ~rules.redundant).sum())} remain after dropping trivial / redundant ones.",
         "Rules were not validated by domain experts or on later data."),
        ("Partial", "Rule mining reports support / confidence without robustness checks across segments or time",
         "Top 20 rules re-tested within each crop; lift reported next to confidence; trivial and redundant rules flagged here.",
         f"Within-crop confidence beats the base rate in {rcv.holds.mean()*100:.0f}% of {len(rcv)} rule × crop tests, but averages {rcv.gap_pts.mean():+.0f} points versus the pooled figure.",
         "No time-split validation; only 20 rules tested; pooled confidence overstates what holds inside each crop."),
        ("Partial", "Studies rely on pre-built datasets, short or coarse histories",
         "Own collection scripts (API-based) and a documented cleaning pipeline.",
         f"{len(panel):,} rows, but only {N_WEEKS} weeks ({pd.Timestamp(WEEKS[0]):%b %Y} – {pd.Timestamp(WEEKS[-1]):%b %Y}).",
         "One annual cycle: seasonality and spike behaviour in other years are unseen."),
        ("Partial", "Forecasts are seldom translated into procurement decisions with stated uncertainty",
         "Procurement Planner tab compares buying now vs later using forecast intervals and optional rule triggers.",
         "Planner shows cost deltas with 80 % whiskers; rule triggers display base rate → conditional probability.",
         "No cost-saving backtest; the earlier '8–12 % savings' claim is unsupported by repo analysis."),
        ("Open", "Association is read as causation, or confounders are ignored",
         "Caveats are stated, but no causal design (e.g. instruments, event studies, panel fixed effects) is used.",
         "Rules are co-occurrence statistics over pooled mandi-weeks that share weather and season.",
         "Add mandi fixed effects or a difference-in-differences style study of heavy-rain events."),
        ("Open", "Mandi-level (hierarchical / spatial) forecasting and price transmission between markets",
         "SARIMA is fitted on the national mean per crop; the panel itself is not used for forecasting.",
         f"{N_MANDI} mandis × 3 crops of series are available but only 3 national series are forecast.",
         "Try hierarchical reconciliation or a panel model with spatial lags."),
        ("Open", "Weather is measured at a point (district HQ), not in the catchment that supplies the mandi",
         "Weather is matched by district HQ coordinates (Open-Meteo).",
         "Distance between mandi town and growing area is not modelled.",
         "Use catchment-weighted rainfall or satellite-derived indices."),
    ]
    colr = {"Addressed": "#2f9e44", "Partial": "#f08c00", "Open": "#e03131"}
    flt = W.ToggleButtons(options=["All", "Addressed", "Partial", "Open"], description="Show")
    o = out()
    def render():
        rows = [g for g in GAPS if flt.value in ("All", g[0])]
        trs = "".join(f"<tr><td>{i+1}</td><td><span class='ap-badge' style='background:{colr[s]}'>{s}</span></td><td><b>{gap}</b></td><td>{how}</td><td>{ev}</td><td>{lim}</td><td style='opacity:.6;font-size:11px'>See literature review</td></tr>"
                      for i, (s, gap, how, ev, lim) in enumerate(GAPS) if (s, gap, how, ev, lim) in rows)
        cnt = {k: sum(1 for g in GAPS if g[0] == k) for k in colr}
        show(o, kpis([("Addressed", cnt["Addressed"], "gaps the project closes", "#2f9e44"), ("Partial", cnt["Partial"], "closed with caveats", "#f08c00"), ("Open", cnt["Open"], "future work", "#e03131")]),
             HTML("<table class='ap-t'><tr><th>#</th><th>Status</th><th>Gap in typical literature</th><th>How AgriProcure responds</th><th>Evidence (computed live)</th><th>Remaining limitation / next step</th><th>References / Next Step</th></tr>" + trs + "</table>"))
    watch([flt], render); render()
    note = callout("<b>Note on sources.</b> The gap statements summarise recurring themes in agricultural price-forecasting and rule-mining work; they are <i>not</i> tied to specific papers. "
                   "Add verified references in the last column (or ask for a sourced literature review) before using this table in a report. The 'Evidence' column is calculated from your repo files each time you run the notebook.", "warn")
    return W.VBox([note, flt, o])

# ── TAB 12 · Export ──────────────────────────────────────────────────────────────────────
def build_export():
    o = out()
    bts = {"Rules (current filters)": W.Button(description="Rules (current filters) → CSV", icon="download"), "Rules fired in scorer": W.Button(description="Scorer matches → CSV", icon="download"),
           "Market weekly": W.Button(description="Market Explorer weekly series → CSV", icon="download"), "Forecast metrics": W.Button(description="Forecast metrics → CSV", icon="download"),
           "Planner": W.Button(description="Planner table → CSV", icon="download"), "HTML": W.Button(description="All current charts → one HTML file", button_style="success", icon="file")}
    keys = {"Rules (current filters)": "rules_filtered", "Rules fired in scorer": "scorer_matches", "Market weekly": "market_weekly", "Forecast metrics": "forecast_metrics", "Planner": "planner_table"}
    def save_csv(name):
        df = STATE.get(keys[name])
        if df is None or len(df) == 0: show(o, callout("Nothing to export yet: open that tab first.", "warn")); return
        p = EXPORT / f"{keys[name]}.csv"; df.to_csv(p, index=False); show(o, callout(f"Saved <code>{p}</code> ({len(df)} rows)", "good"))
    def save_html(_):
        if not FIGS: show(o, callout("No charts yet.", "warn")); return
        parts = [f"<h2>{k}</h2>" + f.to_html(full_html=False, include_plotlyjs="cdn" if i == 0 else False) for i, (k, f) in enumerate(FIGS.items())]
        p = EXPORT / "agriprocure_dashboard_snapshot.html"; p.write_text("<html><meta charset='utf-8'><body style='font-family:sans-serif;max-width:1100px;margin:auto'>" + "".join(parts) + "</body></html>", encoding="utf-8")
        show(o, callout(f"Saved <code>{p}</code> with {len(FIGS)} charts (static, opens in any browser)", "good"))
    for n in keys: bts[n].on_click(lambda _, n=n: save_csv(n))
    bts["HTML"].on_click(save_html)
    return W.VBox([callout(f"Exports are written to <code>{EXPORT}</code>.", "info"), W.HBox(list(bts.values()), layout=LAYOUT_ROW), o])

## Launch the dashboard

In [ ]:
# ── Assemble & launch ────────────────────────────────────────────────────────────────────
ERRORS = []
def safe(fn):
    try: return fn()
    except Exception as e:
        import traceback; ERRORS.append((fn.__name__, e))
        return W.HTML(f"<div class='ap-call ap-bad'><b>{fn.__name__} failed:</b> {e}<pre>{traceback.format_exc()[-900:]}</pre></div>")

TABS = [("1 Overview", build_overview), ("2 Market Explorer", build_market), ("3 Map & Heatmap", build_map), ("4 Weather & Supply", build_drivers),
        ("5 Forecast Lab", build_forecast), ("6 Direction Models", build_models), ("7 Rules Explorer", build_rules), ("8 Spike-Risk Scorer", build_scorer),
        ("9 Procurement Planner", build_planner), ("10 Data & Methods QA", build_qa), ("11 Literature Gaps", build_gaps), ("12 Export", build_export)]
tab = W.Tab(children=[safe(f) for _, f in TABS])
for i, (t, _) in enumerate(TABS): tab.set_title(i, t)
header = HTML(f"<div class='ap-h1'>🌾 AgriProcure — Mandi Price Analytics Dashboard</div>"
              f"<div class='ap-sub'>{N_MANDI} mandis · {N_STATE} states · tomato / onion / potato · {pd.Timestamp(WEEKS[0]):%d %b %Y} → {pd.Timestamp(WEEKS[-1]):%d %b %Y} · all figures computed live from <code>{ROOT.name}/data</code> and <code>/reports</code></div>")
display(header, tab)
if ERRORS:
    print("Tabs with errors:", [n for n, _ in ERRORS])